<a href="https://colab.research.google.com/github/MalakSalehh/Thesis-datasets/blob/main/CLEAN_TRIALtitled1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai pandas numpy scipy scikit-learn

In [1]:
import re
import time
import json
import numpy as np
import pandas as pd

from openai import OpenAI
from sklearn.metrics import cohen_kappa_score, mean_absolute_error
from scipy.stats import pearsonr

In [21]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.2 MB/s eta 0:00:00


In [8]:
RANDOM_STATE = 42

In [3]:
from openai import OpenAI

client = OpenAI(
    api_key="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",
    base_url="https://api.groq.com/openai/v1"
)

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": "You are an expert essay grader."},
        {"role": "user", "content": "Grade this essay..."}
    ],
    temperature=0.3
)

print(response.choices[0].message.content)

I'm ready to grade the essay. Please provide the essay, and I'll assess it based on the following criteria:

1. Content (40%): Does the essay address the topic effectively? Are the arguments well-supported and logical?
2. Organization (20%): Is the essay well-structured and easy to follow? Are the transitions between paragraphs smooth?
3. Style (20%): Is the writing engaging and clear? Are the sentences varied and concise?
4. Mechanics (20%): Are the grammar, spelling, and punctuation correct?

Please provide the essay, and I'll give you a detailed feedback.


In [4]:

url = "https://raw.githubusercontent.com/MalakSalehh/Thesis-datasets/main/training_set_rel3.tsv"
df = pd.read_csv(url, sep="\t", encoding="latin1")

print(df.shape)
df.head()

(12976, 28)


,essay_id,essay_set,essay,rater1_domain1,rater2_domain1,rater3_domain1,domain1_score,rater1_domain2,rater2_domain2,domain2_score,...,rater2_trait3,rater2_trait4,rater2_trait5,rater2_trait6,rater3_trait1,rater3_trait2,rater3_trait3,rater3_trait4,rater3_trait5,rater3_trait6
0,1,1,"Dear local newspaper, I think effects computer...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",5,4,NaN,9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",4,3,NaN,7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",5,5,NaN,10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,"Dear @LOCATION1, I know having computers has a...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
set1 = df[df["essay_set"] == 1].copy()
set1 = set1[["essay_id", "essay", "domain1_score"]]

set1["essay"] = set1["essay"].astype(str).str.replace("\n", " ", regex=False).str.strip()

print(set1.shape)
set1.head()

(1783, 3)


,essay_id,essay,domain1_score
0,1,"Dear local newspaper, I think effects computer...",8
1,2,"Dear @CAPS1 @CAPS2, I believe that using compu...",9
2,3,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7
3,4,"Dear Local Newspaper, @CAPS1 I have found that...",10
4,5,"Dear @LOCATION1, I know having computers has a...",8


In [6]:

def categorize(score):
    if score <= 5:
        return "Low"
    elif score <= 8:
        return "Medium"
    else:
        return "High"

set1["score_category"] = set1["domain1_score"].apply(categorize)

set1["domain1_score"].value_counts().sort_index()

,count
domain1_score,
2,10
3,1
4,17
5,17
6,110
7,135
8,687
9,334
10,316


In [9]:

cal_low = set1[set1["score_category"] == "Low"].sample(2, random_state=RANDOM_STATE)
cal_med = set1[set1["score_category"] == "Medium"].sample(2, random_state=RANDOM_STATE)
cal_high = set1[set1["score_category"] == "High"].sample(2, random_state=RANDOM_STATE)

calibration = pd.concat([cal_low, cal_med, cal_high]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
calibration


,essay_id,essay,domain1_score,score_category
0,1592,The computers are cool. Do you now I werpsite ...,2,Low
1,995,"I think computers are good, because some peopl...",4,Low
2,686,"Dear local newspaper, I would like to speak up...",9,High
3,1572,"Dear to @CAPS1 this @MONTH1 concern, Computers...",8,Medium
4,1567,"Dear @ORGANIZATION1, @DATE1, everywhere you lo...",11,High
5,146,Dear local newspaper I think that usieng compu...,6,Medium


In [17]:
calibration_ids = calibration["essay_id"].tolist()

pool = set1[~set1["essay_id"].isin(calibration_ids)].copy().reset_index(drop=True)

demo_5 = pool.sample(5, random_state=RANDOM_STATE).reset_index(drop=True)

remaining_after_5 = pool[~pool["essay_id"].isin(demo_5["essay_id"])].copy().reset_index(drop=True)
demo_15 = remaining_after_5.sample(15, random_state=RANDOM_STATE).reset_index(drop=True)

print("Calibration:", calibration.shape)
print("Demo 5:", demo_5.shape)
print("Demo 15:", demo_15.shape)

Calibration: (6, 4)
Demo 5: (5, 4)
Demo 15: (15, 4)


In [11]:
# =========================
# RUBRIC
# =========================

rubric_text = """
Essay Prompt:
Write a letter to your local newspaper stating your opinion on the effects computers have on people and persuade readers to agree with your position.

Rubric:
1 – Undeveloped response with minimal support.
2 – Underdeveloped response with weak reasoning.
3 – Minimally developed argument with limited support.
4 – Adequate argument with some supporting details.
5 – Developed argument with clear reasoning and organization.
6 – Strong persuasive response with well-developed ideas.

Scoring rule:
Two raters assign scores from 1–6.
Final score = sum of both scores.
Final score range: 2–12.
"""

In [12]:
# =========================
# CALIBRATION FORMATTERS
# =========================

def format_calibration_examples(calibration_df):
    examples = ""
    for i, row in calibration_df.iterrows():
        examples += f"""
Calibration Example {i+1}

Essay:
{row['essay']}

Human Score: {row['domain1_score']}
Category: {row['score_category']}
"""
    return examples


def format_calibration_examples_with_id(calibration_df):
    examples = []
    for i, row in enumerate(calibration_df.itertuples(index=False), start=1):
        examples.append(f"""
Calibration Example {i}

Essay ID: {row.essay_id}

Essay:
{row.essay}

Human Score: {row.domain1_score}
Category: {row.score_category}
""")
    return "\n".join(examples)

In [13]:
# create both calibration text versions exactly like the notebook styles
calibration_text = format_calibration_examples(calibration)
calibration_text_with_id = format_calibration_examples_with_id(calibration)

In [14]:
def build_full_prompt(essay_text, rubric_text, calibration_text):
    return f"""
You are an expert essay grader evaluating student essays.

{rubric_text}

Calibration Examples:
{calibration_text}

Now evaluate the following essay.

Essay:
{essay_text}

Instructions:
1. Predict the final score between 2 and 12.
2. Predict the category: Low, Medium, or High.
3. Explain your reasoning.

Return your answer in this format:

Final Score:
Category:
Reasoning:
"""

In [22]:
from groq import Groq

client = Groq(api_key="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")

def get_response(prompt, model="llama-3.1-8b-instant"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,
        max_tokens=1024
    )
    return response.choices[0].message.content

In [23]:
essay_text =demo_5 .loc[0, "essay"]
prompt = build_full_prompt(essay_text, rubric_text, calibration_text)
output = get_response(prompt)
print(output)

Final Score: 8
Category: Medium
Reasoning:

This essay demonstrates a clear and organized argument in favor of computers, highlighting their benefits in communication, research, and education. The writer provides specific examples, such as using web chat to reconnect with a distant family member and using online dictionaries to aid in homework. The essay also acknowledges potential drawbacks, such as spending too much time on computers, but frames this as a personal choice rather than a inherent flaw.

The writer's use of transitional phrases and sentences helps to connect their ideas and create a cohesive argument. However, the essay could benefit from more nuanced and detailed explanations of the benefits and drawbacks of computers. Additionally, some of the sentences are quite short and lack specific examples to support the writer's claims.

Overall, the essay demonstrates a clear and well-organized argument, but could benefit from more development and detail to reach a higher score